# 1. Two Sum

**Difficulty:** Easy  
**Topics:** Arrays, hash tables  
**Language:** Python 3  
**Problem:** [LeetCode — Two Sum](https://leetcode.com/problems/two-sum/description/)

Find two indices whose values add up to a target. This solution develops a direct pair search, then improves it using a hash table. Each approach includes the reasoning, a short dry run, a correctness argument, complexity analysis, and a complete LeetCode-compatible implementation.

Run the notebook from top to bottom. It requires only Python's standard library. To submit, copy the final code cell. To experiment, edit the inputs in **Try your own input**.

## 1. Problem

### Statement summary

Given an integer list `nums` and an integer `target`, return the zero-based indices of two distinct positions whose values add up to `target`. Exactly one valid index pair is guaranteed. Either order of the two indices is acceptable.

### Constraints

- `2 <= len(nums) <= 10_000`
- `-10**9 <= nums[i] <= 10**9`
- `-10**9 <= target <= 10**9`
- Exactly one valid pair exists.

### Examples

**Example 1**

```text
Input: nums = [2, 7, 11, 15], target = 9
Output: [0, 1]
```

The values at positions 0 and 1 are 2 and 7; together they make 9.

**Example 2**

```text
Input: nums = [3, 2, 4], target = 6
Output: [1, 2]
```

The matching values are 2 and 4, at positions 1 and 2. The 3 at position 0 cannot be used twice.

**Example 3**

```text
Input: nums = [3, 3], target = 6
Output: [0, 1]
```

Equal values are allowed when they occupy different positions.

**Follow-up:** Find an approach with better time complexity than O(n²).

## 2. Understand the task

The answer contains **indices**, not the two values. For `[2, 7, 11, 15]` with target `9`, return `[0, 1]`, not `[2, 7]`.

The list is not guaranteed to be sorted, and values may be negative or repeated. A position cannot pair with itself. Sorting values alone would lose their original positions, so the approaches below work directly with the original list.

A useful way to restate the condition is:

```text
nums[i] + nums[j] = target
nums[j] = target - nums[i]
```

The first form suggests trying pairs. The second suggests remembering previously encountered values so a partner can be found quickly.

In [1]:
from typing import List

# Shared input for the dry runs.
EXAMPLE_NUMS = [2, 7, 11, 15]
EXAMPLE_TARGET = 9

## 3. Approach 1 — brute force

### Idea

Choose an index `i`, then check every later index `j`. Return immediately when the two values sum to the target.

### Algorithm

1. Iterate `i` over the list.
2. Iterate `j` from `i + 1` to the end.
3. If `nums[i] + nums[j] == target`, return `[i, j]`.

Starting `j` at `i + 1` prevents reusing a position and avoids checking a pair twice.

### Dry run

`nums = [2, 7, 11, 15]`, `target = 9`

| Check | `i` | `j` | Sum | Result |
|---|---:|---:|---|---|
| 1 | 0 | 1 | 2 + 7 = 9 | Return `[0, 1]` |

The first pair succeeds, so there are no further checks. If it failed, `j` would advance; after all later positions were checked, `i` would advance.

### Why it works

Every pair of different indices has an ordering `i < j`. The nested loops visit every such pair unless an answer has already been returned. A returned pair satisfies the sum condition, and its indices differ. Because a valid pair is guaranteed, the search must find it.

### Complexity

- **Time:** O(n²) in the worst case, from up to n(n − 1)/2 pair checks. The example above finishes after one check.
- **Auxiliary space:** O(1), excluding the returned pair.

### Implementation

In [2]:
class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        for i in range(len(nums)):
            for j in range(i + 1, len(nums)):
                if nums[i] + nums[j] == target:
                    return [i, j]
        raise ValueError("No pair sums to target.")

# Preserve this implementation before the next Solution class is defined.
BruteForceSolution = Solution
print(BruteForceSolution().twoSum(EXAMPLE_NUMS, EXAMPLE_TARGET))

[0, 1]


In [3]:
# Teaching trace, separate from the submission implementation.
def trace_brute_force(nums, target):
    checks = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            checks += 1
            total = nums[i] + nums[j]
            print(f"Check {checks}: i={i}, j={j}, {nums[i]} + {nums[j]} = {total}")
            if total == target:
                print(f"Return {[i, j]}; pair checks = {checks}")
                return [i, j]
    raise ValueError("No pair sums to target.")

assert trace_brute_force(EXAMPLE_NUMS, EXAMPLE_TARGET) == [0, 1]

Check 1: i=0, j=1, 2 + 7 = 9
Return [0, 1]; pair checks = 1


## 4. Approach 2 — one-pass hash table

### From repeated search to remembered values

Brute force repeatedly searches for the partner of the current value. Instead, compute the partner as `target - nums[i]` and ask whether it has already appeared.

Use a dictionary `seen` mapping **value → earlier index**. Store indices because those are what the problem asks us to return. The dictionary stores encountered values, not missing complements.

### Algorithm

1. Start with an empty dictionary.
2. For each index `i`, compute `diff = target - nums[i]`.
3. If `diff` is in `seen`, return `[seen[diff], i]`.
4. Otherwise, record `seen[nums[i]] = i` and continue.

**Look up before inserting.** Inserting first could allow the current position to match itself when its value is half the target.

### Dry run

`nums = [2, 7, 11, 15]`, `target = 9`

| Visit | Index | Value | Needed partner | Dictionary before lookup | Action |
|---|---:|---:|---:|---|---|
| 1 | 0 | 2 | 7 | `{}` | No match; store `2: 0` |
| 2 | 1 | 7 | 2 | `{2: 0}` | Return `[0, 1]` |

The algorithm stops after two visits; 11 and 15 are never processed.

### Why it works

Before processing index `i`, the dictionary contains a representative index for every value in the prefix before `i`.

- Initially, the prefix and dictionary are empty.
- If the needed partner exists in the dictionary, it belongs to an earlier position. Its value plus `nums[i]` equals the target, so the returned pair is valid and uses distinct indices.
- If no partner exists, storing the current value preserves the property for the next visit. Overwriting the saved index of a repeated value still leaves an earlier representative for that value.

When the later member of the guaranteed answer pair is visited, the earlier member's value is already present. Therefore, the lookup finds a valid answer.

### Complexity

- **Expected time:** O(n), assuming average O(1) dictionary lookup and insertion.
- **Auxiliary space:** O(n), for stored values and indices.

Pathological dictionary behavior can increase the time bound; the usual analysis uses expected constant-time hash operations.

### Implementation

In [4]:
class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        seen = {}
        for i in range(len(nums)):
            diff = target - nums[i]
            if diff in seen:
                return [seen[diff], i]
            seen[nums[i]] = i
        raise ValueError("No pair sums to target.")

HashMapSolution = Solution
print(HashMapSolution().twoSum(EXAMPLE_NUMS, EXAMPLE_TARGET))

[0, 1]


In [5]:
def trace_hashmap(nums, target):
    seen = {}
    for i in range(len(nums)):
        diff = target - nums[i]
        print(f"Visit {i + 1}: index={i}, value={nums[i]}, need={diff}, seen={seen}")
        if diff in seen:
            result = [seen[diff], i]
            print(f"Return {result}; loop visits = {i + 1}")
            return result
        seen[nums[i]] = i
        print(f"  Store value {nums[i]} at index {i}: seen={seen}")
    raise ValueError("No pair sums to target.")

assert trace_hashmap(EXAMPLE_NUMS, EXAMPLE_TARGET) == [0, 1]

Visit 1: index=0, value=2, need=7, seen={}
  Store value 2 at index 0: seen={2: 0}
Visit 2: index=1, value=7, need=2, seen={2: 0}
Return [0, 1]; loop visits = 2


## 5. Comparison and common mistakes

| Approach | Time | Auxiliary space | Main trade-off |
|---|---|---|---|
| Brute force | O(n²) worst case | O(1) | Simple, but repeats searches |
| One-pass hash table | O(n) expected | O(n) | Uses memory to avoid repeated searches |

Choose the hash-table approach for the final submission. Although brute force finds the first example immediately, its worst-case work grows quadratically. A single small example does not measure how an algorithm scales.

Common mistakes:

- Returning values instead of indices.
- Starting the inner brute-force loop at `i` and allowing one position twice.
- Inserting into `seen` before checking for a partner.
- Using a set when the earlier index must also be recovered.
- Treating two equal values at different positions as an invalid pair.

Both implementations include a `ValueError` for custom inputs without an answer. LeetCode guarantees an answer, so valid platform inputs do not reach that branch. The main algorithms return the first match; uniqueness is validated separately by the manual harness.

## 6. Test the solutions

The first three cases reproduce the problem's example inputs and outputs. Additional cases cover negatives, zero, a pair discovered later, and the integer limits.

For each approach, verify that the result has two distinct in-range indices, that their values sum to the target, and that the input list remains unchanged. Compare pairs without depending on the permitted output order.

In [6]:
def check_result(nums, target, result):
    assert isinstance(result, list) and len(result) == 2
    i, j = result
    assert type(i) is int and type(j) is int
    assert 0 <= i < len(nums) and 0 <= j < len(nums)
    assert i != j, "Cannot reuse an index."
    assert nums[i] + nums[j] == target

TEST_CASES = [
    ("Problem example 1", [2, 7, 11, 15], 9),
    ("Problem example 2", [3, 2, 4], 6),
    ("Problem example 3", [3, 3], 6),
    ("Supplementary: negatives", [-3, 4, 3, 90], 0),
    ("Supplementary: zeros", [0, 4, 3, 0], 0),
    ("Supplementary: later pair", [1, 2, 3, 4], 7),
    ("Supplementary: negative target", [-5, -2, -8], -10),
    ("Supplementary: boundaries", [-10**9, 10**9], 0),
    ("Supplementary: no self-pair", [3, 2, 4], 6),
]
for name, nums, target in TEST_CASES:
    before = nums.copy()
    answers = []
    for implementation in (BruteForceSolution, HashMapSolution):
        result = implementation().twoSum(nums, target)
        check_result(nums, target, result)
        assert nums == before
        answers.append(result)
    assert set(answers[0]) == set(answers[1])
    print(f"PASS | {name}: {answers[1]}")

for implementation in (BruteForceSolution, HashMapSolution):
    try:
        implementation().twoSum([1, 2], 99)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for an input without a solution.")
print("Both implementations also reject a no-solution input.")

PASS | Problem example 1: [0, 1]
PASS | Problem example 2: [1, 2]
PASS | Problem example 3: [0, 1]
PASS | Supplementary: negatives: [0, 2]
PASS | Supplementary: zeros: [0, 3]
PASS | Supplementary: later pair: [2, 3]
PASS | Supplementary: negative target: [1, 2]
PASS | Supplementary: boundaries: [0, 1]
PASS | Supplementary: no self-pair: [1, 2]
Both implementations also reject a no-solution input.


## 7. Try your own input

Edit `nums_text`, `target_text`, and optionally `approach` in the next cell, then run it again. Use JSON list syntax such as `[2, 7, 11, 15]`.

The harness validates the constraints and requires exactly one index pair. Its expected O(n) validation pass is separate from the chosen algorithm. Invalid input produces an explanatory message. No interactive prompt blocks **Run All**.

In [7]:
import json

def validate_problem_input(nums, target):
    if not isinstance(nums, list) or not 2 <= len(nums) <= 10_000:
        raise ValueError("nums must be a list with 2 to 10,000 integers.")
    if any(type(value) is not int or not -10**9 <= value <= 10**9 for value in nums):
        raise ValueError("Each value must be an integer from -10^9 to 10^9.")
    if type(target) is not int or not -10**9 <= target <= 10**9:
        raise ValueError("target must be an integer from -10^9 to 10^9.")
    counts = {}
    pair_count = 0
    for value in nums:
        pair_count += counts.get(target - value, 0)
        counts[value] = counts.get(value, 0) + 1
    if pair_count != 1:
        raise ValueError(f"Expected exactly one index pair; found {pair_count}.")


def run_manual(nums_text, target_text, approach="hashmap"):
    try:
        nums = json.loads(nums_text)
        target = json.loads(target_text)
        validate_problem_input(nums, target)
        implementations = {"brute_force": BruteForceSolution, "hashmap": HashMapSolution}
        if approach not in implementations:
            raise ValueError("approach must be 'brute_force' or 'hashmap'.")
        result = implementations[approach]().twoSum(nums, target)
        check_result(nums, target, result)
        i, j = result
        print(f"{approach}: indices {result}; {nums[i]} + {nums[j]} = {target}")
        return result
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then run this cell again.
nums_text = "[2, 7, 11, 15]"
target_text = "9"
approach = "hashmap"  # Or "brute_force".
manual_result = run_manual(nums_text, target_text, approach)

hashmap: indices [0, 1]; 2 + 7 = 9


In [8]:
# Supplementary checks of the manual harness.
assert run_manual("[3, 3]", "6", "brute_force") == [0, 1]
assert run_manual("[1, 2]", "99") is None
assert run_manual("[1, 1, 1]", "2") is None
assert run_manual("[true, 2]", "3") is None
assert run_manual("not JSON", "9") is None
assert run_manual("[2, 7]", "9.0") is None
assert run_manual("[2, 7]", "9", "unknown") is None
print("Manual-input harness checks passed.")

brute_force: indices [0, 1]; 3 + 3 = 6
Input error: Expected exactly one index pair; found 0.
Input error: Expected exactly one index pair; found 3.
Input error: Each value must be an integer from -10^9 to 10^9.
Input error: Expecting value: line 1 column 1 (char 0)
Input error: target must be an integer from -10^9 to 10^9.
Input error: approach must be 'brute_force' or 'hashmap'.
Manual-input harness checks passed.


## 8. Final submission

Copy the following cell into LeetCode's Python 3 editor. It implements the one-pass hash-table approach using the required `Solution.twoSum(self, nums, target)` interface. The import also makes the code self-contained outside LeetCode.

In [9]:
from typing import List

class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        seen = {}
        for i in range(len(nums)):
            diff = target - nums[i]
            if diff in seen:
                return [seen[diff], i]
            seen[nums[i]] = i
        raise ValueError("No pair sums to target.")